# Handwritten Digit Classification — ML Pipeline
**African Leadership University | BSE | Machine Learning Operations**

This notebook covers the full ML cycle:
1. Data Acquisition & Exploration
2. Preprocessing
3. Model Training (Transfer Learning — MobileNetV2)
4. Model Evaluation (all metrics)
5. Prediction Function
6. Model Retraining Trigger

In [ ]:
import os, sys, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import tensorflow as tf
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, accuracy_score,
                              precision_score, recall_score, f1_score)

print('TensorFlow:', tf.__version__)
print('Root:', ROOT)

## 1. Data Acquisition & Exploration

In [ ]:
from src.preprocessing import CLASSES, CLASS_TO_IDX, IDX_TO_CLASS, IMG_SIZE

DATASET_DIR = os.path.join(ROOT, 'Handwritten_Dataset')
print('Classes:', CLASSES)

counts = {}
for cls in CLASSES:
    cls_dir = os.path.join(DATASET_DIR, cls)
    imgs = [f for f in os.listdir(cls_dir) if f.endswith('.jpg') and '- Copy' not in f]
    counts[cls] = len(imgs)
    print(f'  {cls}: {len(imgs)} images')
print(f'\nTotal: {sum(counts.values())} images')

### Feature 1: Class Distribution
Shows how many images exist per digit class. A balanced dataset means the model trains without class bias.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(counts.keys(), counts.values(), color='steelblue')
ax.set_title('Training Images per Class')
ax.set_ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print('Interpretation: The dataset is roughly balanced (~16 images/class), so the model trains without class bias.')

### Feature 2: Sample Images per Class
Displays one real handwritten image per digit to show natural variation in stroke style, thickness, and angle.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for ax, cls in zip(axes.flatten(), CLASSES):
    cls_dir = os.path.join(DATASET_DIR, cls)
    imgs = [f for f in os.listdir(cls_dir) if f.endswith('.jpg') and '- Copy' not in f]
    img = mpimg.imread(os.path.join(cls_dir, imgs[0]))
    ax.imshow(img)
    ax.set_title(cls, fontsize=11)
    ax.axis('off')
plt.suptitle('Sample Images per Class', fontsize=13)
plt.tight_layout()
plt.show()
print('Interpretation: Real handwritten digits show high variation in style — this is why transfer learning and augmentation are needed.')

### Feature 3: Average Pixel Brightness per Class
Measures how dark or light each digit class is on average.

In [ ]:
avg_brightness = {}
for cls in CLASSES:
    cls_dir = os.path.join(DATASET_DIR, cls)
    vals = []
    for fname in os.listdir(cls_dir):
        if fname.endswith('.jpg') and '- Copy' not in fname:
            img = np.array(Image.open(os.path.join(cls_dir, fname)).convert('L').resize((64, 64)), dtype=np.float32)
            vals.append(img.mean())
    avg_brightness[cls] = np.mean(vals)

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(avg_brightness.keys(), avg_brightness.values(), color='mediumseagreen')
ax.set_title('Average Pixel Brightness per Class')
ax.set_ylabel('Mean Intensity (0-255)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print('Interpretation: Digits like "1" are brighter (less ink) while "8" and "0" are darker (more ink coverage).')

### Feature 4: Raw Image Size Distribution
Shows the spread of original image dimensions before preprocessing.

In [ ]:
widths, heights = [], []
for cls in CLASSES:
    cls_dir = os.path.join(DATASET_DIR, cls)
    for fname in os.listdir(cls_dir):
        if fname.endswith('.jpg') and '- Copy' not in fname:
            img = Image.open(os.path.join(cls_dir, fname))
            widths.append(img.width)
            heights.append(img.height)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(widths, bins=15, color='coral')
axes[0].set_title('Image Width Distribution')
axes[0].set_xlabel('Pixels')
axes[1].hist(heights, bins=15, color='orchid')
axes[1].set_title('Image Height Distribution')
axes[1].set_xlabel('Pixels')
plt.tight_layout()
plt.show()
print(f'Width  - min: {min(widths)}, max: {max(widths)}, mean: {int(np.mean(widths))}')
print(f'Height - min: {min(heights)}, max: {max(heights)}, mean: {int(np.mean(heights))}')
print('Interpretation: Images vary significantly in size. Resizing to 96x96 ensures consistent CNN input.')

## 2. Preprocessing

In [ ]:
from src.preprocessing import load_dataset

TRAIN_DIR = os.path.join(ROOT, 'data', 'train')
TEST_DIR  = os.path.join(ROOT, 'data', 'test')

X_train, y_train = load_dataset(TRAIN_DIR)
X_test,  y_test  = load_dataset(TEST_DIR)

print(f'Train: {X_train.shape}, labels: {y_train.shape}')
print(f'Test:  {X_test.shape},  labels: {y_test.shape}')
print(f'Pixel range: [{X_train.min():.2f}, {X_train.max():.2f}]  (MobileNetV2 scale: -1 to 1)')

In [ ]:
# Show preprocessed images (inverted + scaled for MobileNetV2)
# Display by converting back to 0-1 range for visualization
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
shown = {}
for img, label in zip(X_train, y_train):
    if label not in shown:
        shown[label] = img
    if len(shown) == 10:
        break

for ax, (label, img) in zip(axes.flatten(), sorted(shown.items())):
    ax.imshow((img + 1.0) / 2.0)  # convert -1..1 back to 0..1 for display
    ax.set_title(IDX_TO_CLASS[label])
    ax.axis('off')
plt.suptitle(f'Preprocessed Images ({IMG_SIZE[0]}x{IMG_SIZE[1]}, inverted, MobileNetV2 scaled)')
plt.tight_layout()
plt.show()

## 3. Model Training (Transfer Learning — MobileNetV2)

In [ ]:
from src.model import build_model, augment_dataset

X_aug, y_aug = augment_dataset(X_train, y_train, factor=15)
idx = np.random.permutation(len(X_aug))
X_aug, y_aug = X_aug[idx], y_aug[idx]
print(f'Original training set: {X_train.shape}')
print(f'Augmented training set: {X_aug.shape}  (15x augmentation)')

model = build_model()
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(patience=8, restore_best_weights=True, verbose=1)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

# Phase 1: frozen base — train only top layers
print('=== Phase 1: Training top layers (base frozen) ===')
history1 = model.fit(
    X_aug, y_aug,
    epochs=20,
    validation_split=0.15,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# Phase 2: unfreeze top 30 layers and fine-tune
print('=== Phase 2: Fine-tuning top 30 base layers ===')
base_model = model.layers[1]
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop2 = EarlyStopping(patience=10, restore_best_weights=True, verbose=1)
history2 = model.fit(
    X_aug, y_aug,
    epochs=50,
    validation_split=0.15,
    callbacks=[early_stop2, reduce_lr],
    verbose=1
)

In [ ]:
# Training curves for both phases
for history, title in [(history1, 'Phase 1 - Frozen Base'), (history2, 'Phase 2 - Fine-tuning')]:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_title(f'{title} - Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_title(f'{title} - Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

MODEL_SAVE_PATH = os.path.join(ROOT, 'models', 'handwritten_digit_model.h5')
os.makedirs(os.path.join(ROOT, 'models'), exist_ok=True)
model.save(MODEL_SAVE_PATH)
print(f'Model saved to {MODEL_SAVE_PATH}')

## 4. Model Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_acc*100:.2f}%')

y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

print('\nPer-class Classification Report:')
print(classification_report(y_test, y_pred, target_names=CLASSES))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(9, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
plt.title('Confusion Matrix - Test Set')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
metrics = {
    'Accuracy':  accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, average='weighted'),
    'Recall':    recall_score(y_test, y_pred, average='weighted'),
    'F1-Score':  f1_score(y_test, y_pred, average='weighted'),
}
df_metrics = pd.DataFrame(list(metrics.items()), columns=['Metric', 'Score'])
df_metrics['Score'] = df_metrics['Score'].map('{:.4f}'.format)
print(df_metrics.to_string(index=False))

## 5. Prediction Function

In [ ]:
from src.prediction import predict

test_images = []
for cls in CLASSES:
    cls_dir = os.path.join(TEST_DIR, cls)
    if os.path.isdir(cls_dir):
        for f in os.listdir(cls_dir):
            if f.endswith('.jpg'):
                test_images.append((os.path.join(cls_dir, f), cls))

samples = random.sample(test_images, min(5, len(test_images)))
fig, axes = plt.subplots(len(samples), 2, figsize=(10, 3 * len(samples)))

for i, (img_path, true_label) in enumerate(samples):
    result = predict(img_path, model=model)
    correct = result['predicted_class'] == true_label

    axes[i][0].imshow(mpimg.imread(img_path))
    axes[i][0].set_title(f'True: {true_label}  |  Pred: {result["predicted_class"]} {"OK" if correct else "WRONG"}')
    axes[i][0].axis('off')

    probs = result['all_probabilities']
    colors = ['green' if k == result['predicted_class'] else 'steelblue' for k in probs]
    axes[i][1].barh(list(probs.keys()), [v * 100 for v in probs.values()], color=colors)
    axes[i][1].set_xlabel('Confidence (%)')
    axes[i][1].set_title(f"Confidence: {result['confidence']*100:.1f}%")

plt.tight_layout()
plt.show()

## 6. Model Retraining Trigger

In [ ]:
from src.model import train as train_model

print('Starting retraining...')
retrained_model = train_model(
    train_dir=TRAIN_DIR,
    epochs=50,
    save_path=MODEL_SAVE_PATH
)
print('Retraining complete. Model saved.')

In [ ]:
loss, acc = retrained_model.evaluate(X_test, y_test, verbose=0)
y_pred_retrained = np.argmax(retrained_model.predict(X_test, verbose=0), axis=1)
print(f'Retrained model - Test Accuracy: {acc*100:.2f}%')
print(f'Retrained model - F1-Score: {f1_score(y_test, y_pred_retrained, average="weighted"):.4f}')